# INFO 159/259
#<center> Homework 6: Unsupervised Methods </center>

<center> Due: April 30, 2026 @ 11:59pm </center>


In class, we have covered several unsuperivsed methods for working with data. In this final homework, we want you to tell us something interesting about a dataset with any of the unsuperivised methods we covered during class.

## Dataset
You can pick any of the datasets below for your analysis:

 <a name="cell-id"></a>
    [CMU Book Summary Dataset](https://www.cs.cmu.edu/~dbamman/booksummaries.html)
        Metadata: Title, author, publication date, genre

 <a name="cell-id"></a>
    [CMU Movie Summary Dataset](http://www.cs.cmu.edu/~ark/personas/)
        Metadata: Movie box office revenue, genre, release date, runtime, and language

 <a name="cell-id"></a>
    [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/)
        Metadata: positive/negative sentiment

 <a name="cell-id"></a>
    [Congressional Speech Data](https://www.cs.cornell.edu/home/llee/data/convote.html)
        Metadata: political party

 <a name="cell-id"></a>
    [100K ArXiv abstracts](https://drive.google.com/file/d/1ThK1D9AstYI6s2Z7m9SmvLqLZPneMp12/view?usp=sharing)
        Metadata: ArXiv subject (e.g., math, physics, cs)


## Deliverable 1: Tell us about the dataset you picked.
In the text box below, tell us which dataset you selected and why you are interested in it. Describe the metadata of your selected dataset.

(your response goes here)

## Deliverable 2: Tell us about the methods you chose


For this assignment, you will use two methods from class to explore your data and tell us something interesting about it.

a.) You must use one of the following methods:

* Topic modeling
* Hierarchical clustering
* K-means clustering

b.) In addition, you should use one more method from the following options:

* PMI
* dependency parsing
* NER
* WSD (using WordNet)
* Relation extration
* Classification (e.g. logistic regression, BERT)
* Word embeddings (word2vec, contextual embeddings)

You are free to use any external resources and libraries for this assignment; the goal is to put use your comprehensive knowledge of NLP to use.

As starting points, you can use `spacy` for parsing/NER, `lda` for topic modeling, `sklearn` for clustering (KMeans, TF-IDF). You can implement methods for which you cannot find libraries.

In the text box below, tell us about:
* which unsupervised method you chose and why you chose that method for this particular dataset (1 sentence)
* what pre-processing steps you had to perform to run the method on your data (up to 2 sentences)
* how you choose to represent your documents if using clustering methods (up to 2 sentences)


## Deliverable 3: Code for analysis
In the code block below, add your code for analysis. You can organize your code in way that fits best for your analysis, including adding more code blocks.

In [76]:
#read/clean movie metadata text file and make dictionaries
#numeric id is clean six digits
#freebase id has slashes and letters
title_to_numeric = {}
freebase_to_numeric = {}
with open("movie.metadata.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        numeric_id = parts[0]
        freebase_id = parts[1]
        title = parts[2]
        title_to_numeric[title] = numeric_id #title: 123456
        freebase_to_numeric[freebase_id] = numeric_id #m/0ab1c3: 123456

In [70]:
#read/clean tv tropes text file and make dictionary structured like trope: movie titles
import json
trope_mov_pairs = {} #empty dictonary to be structured as trope: movie1, movie2, etc.
with open("tvtropes.clusters.txt", "r", encoding="utf-8") as f:
  for line in f:
    line = line.strip() #clean newline
    parts = line.split("\t") #clean tabs and convert into list
    trope = parts[0] #index 0 is the trope
    data = json.loads(parts[1]) #text into dictionary (keys are character, movie, id, actor)
    title = data["movie"]
    if trope in trope_mov_pairs:
      if title not in trope_mov_pairs[trope]:
        trope_mov_pairs[trope].append(title)
    else: trope_mov_pairs[trope] = [title]
print(trope_mov_pairs)

{'absent_minded_professor': ['Flubber', 'Richie Rich', 'The Shadow', 'Them!', 'Stargate'], 'adventurer_archaeologist': ['Indiana Jones and the Kingdom of the Crystal Skull', 'Indiana Jones and the Raiders of the Lost Ark', 'Indiana Jones and the Temple of Doom', 'The Mummy'], 'arrogant_kungfu_guy': ['Enter the Dragon', 'The Karate Kid', 'Crouching Tiger, Hidden Dragon', 'Kill Bill Volume 2', 'Rocky', 'Rocky III', 'G.I. Joe: The Rise of Cobra', 'Highlander', 'Star Wars Episode III: Revenge of the Sith'], 'big_man_on_campus': ['Never Been Kissed', 'John Tucker Must Die', "It's a Boy Girl Thing", "National Lampoon's Van Wilder", 'Election', 'High School Musical', "Can't Hardly Wait"], 'bounty_hunter': ['For a Few Dollars More', 'The Outlaw Josey Wales', 'The Proposition', 'The Rundown', 'Blade Runner', 'Raising Arizona', 'Midnight Run', 'The Bounty Hunter', 'The Chronicles of Riddick', 'Wanted: Dead or Alive'], 'brainless_beauty': ["Fool's Gold", 'The Opposite of Sex', 'Election', 'The Ho

In [71]:
#read/clean plot summaries text file and make dictioanry structured like id: summary
movie_plots = {}
with open("plot_summaries.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t", 1)
        id, plot = parts
        movie_plots[id] = plot

In [77]:
docs = {}
for trope, ids in trope_mov_pairs.items():
    summaries = []
    for movie_id in ids:
        numeric_id = None
        # case 1: already numeric
        if movie_id in movie_plots:
            numeric_id = movie_id
        # case 2: Freebase ID
        elif movie_id in freebase_to_numeric:
            numeric_id = freebase_to_numeric[movie_id]
        # case 3: title
        elif movie_id in title_to_numeric:
            numeric_id = title_to_numeric[movie_id]
        if numeric_id is None:
            continue
        plot = movie_plots.get(numeric_id)
        if plot:
            summaries.append(plot)
    docs[trope] = " ".join(summaries)
print(docs)

{'absent_minded_professor': 'Professor Philip Brainard , a professor at Medfield College, is developing a new energy source in an attempt to raise enough money to save the college from closure. His preoccupancy with his research distracts him from his fiancée and the president of Medfield College Doctor Sara Jean Reynolds ; he has missed two weddings in the past as a result of this, much to the anger of Sara. On the day of the third attempted wedding, Philip is approached by his former partner Wilson Croft , who has profited from ideas he has stolen from Brainard and now desires to steal Sara from Philip and make her his wife, which he declares directly to Philip. Before he can make it to the wedding, his latest experiment shows fast development, forcing him to miss another wedding. The resulting substance created from the experiment is a green slime that proves to be difficult to control and wreaks havoc on the neighborhood before Brainard finally manages to capture him. Weebo , Phili

## Deliverable 4: Results and Analysis
Here, use code or text blocks below to reflect on interesting results from your analysis. You can use graphs or figures from your code above.

Additionally, **add your interpretation of the results**. What did you find interesting from what you obsereved?

# Submission
Congratulations on finishing HW6! Please ensure that you submit this completed notebook onto Gradescope. Make sure all cells in the notebook are run so that print statements are visible.

`File` --> `Download` --> `Download .ipynb`

Make sure your notebook is named `HW6.ipynb` when you upload it to [Gradescope](https://www.gradescope.com/courses/1238346/) before April 30, 2026 at 11:59pm.